[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/milioe/casos-ia-ibero-diplomado/blob/main/Modulo%204%3A%20NLP/10_Word2Vec_scaling.ipynb)


# Word2Vec a escala — de un cuento a un corpus en español

En el notebook **09** vimos la idea de Word2Vec (a mano con Keras y con un corpus pequeño). Aquí hacemos el salto práctico con **`gensim`**, la librería que implementa Word2Vec de verdad (CBOW, skip-gram, negative sampling).

Vamos en dos partes:

1. **Parte 1 — Un solo cuento (Borges, *Funes el memorioso*)**: extraemos texto de un PDF, tokenizamos palabra por palabra, entrenamos Word2Vec y visualizamos su vocabulario en 3D con PCA.
2. **Parte 2 — Corpus completo en español**: cargamos **todos** los fragmentos `spanishText_*.zip` del corpus Kaggle (descarga opcional desde Drive), entrenamos CBOW y Skip-gram, exploramos vecinos, analogías, suma/resta de vectores y PCA 3D.

**La enseñanza central (The Bitter Lesson):** por muy limpio que limpiemos el texto, lo que más mueve la calidad es el **volumen de datos**. Word2Vec con un cuento enseña el pipeline; Word2Vec con millones de palabras enseña semántica útil. Para entender tu corpus a fondo, generar texto largo o comparar PDFs enteros, más adelante veremos **transformers** (BERT, GPT).


## Setup — instalar librerías

Necesitamos:

- `gensim` — implementación eficiente de Word2Vec
- `pypdf` — leer PDFs digitales
- `scikit-learn` — PCA para visualización
- `plotly` — gráficos 3D interactivos


In [ ]:
%pip install -q numpy matplotlib pandas scikit-learn gensim pypdf plotly gdown

In [ ]:
import os
import re
import zipfile
import random
import pickle
from pathlib import Path
from collections import Counter

import gdown

import numpy as np
import matplotlib.pyplot as plt
import plotly.graph_objects as go
from pypdf import PdfReader
from gensim.models import Word2Vec
from sklearn.decomposition import PCA

np.random.seed(42)
random.seed(42)
plt.rcParams.update({"figure.dpi": 100, "axes.grid": True, "grid.alpha": 0.3})

# Stopwords en español (misma lista didáctica que en 02-PDF_reporte.ipynb)
STOP = {
    "el", "la", "los", "las", "un", "una", "unos", "unas", "y", "o", "u", "de", "del", "al",
    "en", "que", "con", "por", "para", "como", "se", "su", "sus", "lo", "le", "les", "a",
    "no", "si", "ya", "más", "menos", "tan", "entre", "sobre", "ser", "es", "son", "fue",
    "ha", "han", "he", "hay", "está", "están", "este", "esta", "esto", "ese", "esa", "eso",
}

## Verificar archivos

**Parte 1 (Borges):** `borges_funes.pdf` en esta carpeta.

**Parte 2 (corpus Kaggle):** los `.zip` `spanishText_*.zip` en `corpus_descargado/` (la celda siguiente puede descargar los que falten desde [Google Drive](https://drive.google.com/drive/folders/1U9V2Wp-ccTAz7JP__FxU1uAF79by-9uf?usp=sharing)).

Si ya tienes un zip suelto (`spanishText_10000_15000.zip`) o archivos en `corpus_kaggle/`, también se detectan.


In [ ]:
BASE_DIR = Path(".")
PDF_PATH = BASE_DIR / "borges_funes.pdf"
CORPUS_DIR = BASE_DIR / "corpus_descargado"
CORPUS_DIR.mkdir(exist_ok=True)

def listar_zips_corpus():
    """Busca spanishText_*.zip en corpus_descargado, corpus_kaggle y la carpeta del módulo."""
    vistos, zips = set(), []
    for carpeta in (CORPUS_DIR, BASE_DIR / "corpus_kaggle", BASE_DIR):
        if not carpeta.exists():
            continue
        for z in sorted(carpeta.glob("spanishText_*.zip")):
            if z.name not in vistos:
                vistos.add(z.name)
                zips.append(z)
    return zips

print("PDF existe:", PDF_PATH.exists())
print(f"ZIPs del corpus encontrados: {len(listar_zips_corpus())}")
for z in listar_zips_corpus()[:5]:
    print(f"  - {z.name}")
if len(listar_zips_corpus()) > 5:
    print(f"  ... y {len(listar_zips_corpus()) - 5} más")

# Parte 1 — Un cuento: Borges → tokens → Word2Vec → PCA 3D

Flujo completo:

1. Extraer texto del PDF
2. Limpiar y tokenizar (palabra por palabra)
3. Entrenar Word2Vec con gensim
4. Explorar vocabulario
5. Buscar palabras parecidas
6. Sumar y restar vectores
7. Visualizar en 3D con PCA


## Extraer texto del PDF

Este PDF ya trae texto digital, no es una foto escaneada. Si tuvieras solo una imagen escaneada, antes habría que pasar OCR (como en el notebook 03 del módulo).

`pypdf` lee el PDF y devuelve el texto página por página.


In [ ]:
def extraer_texto_pdf(ruta_pdf):
    reader = PdfReader(str(ruta_pdf))
    partes = []
    for pagina in reader.pages:
        texto = pagina.extract_text()
        if texto:
            partes.append(texto)
    return "\n".join(partes)

texto_borges = extraer_texto_pdf(PDF_PATH)
print(f"Caracteres extraídos: {len(texto_borges):,}")
print("\nPrimeras líneas:")
print(texto_borges[:350])

## Tokenizar: una palabra = un token

Word2Vec trabaja con listas de palabras. Cada oración es una lista de tokens.

Normalizamos a minúsculas y solo dejamos letras del español (incluyendo acentos y ñ). Luego partimos el texto en oraciones aproximadas.

**Importante:** Word2Vec da **un vector por palabra**. Si quieres representar `"funes memorioso"` como un solo concepto, tendrías que tratarlo como un token único (`"funes_memorioso"`) o sumar/promediar vectores después.


In [ ]:
def limpiar_y_tokenizar(texto, min_tokens=3):
    texto = texto.lower()
    texto = re.sub(r"[^a-záéíóúüñ\n ]+", " ", texto)
    trozos = re.split(r"[.!?\n]+", texto)
    oraciones = []
    for trozo in trozos:
        trozo = re.sub(r"\s+", " ", trozo).strip()
        if not trozo:
            continue
        tokens = trozo.split()
        if len(tokens) >= min_tokens:
            oraciones.append(tokens)
    return oraciones

In [ ]:
oraciones_borges = limpiar_y_tokenizar(texto_borges)
n_palabras = sum(len(o) for o in oraciones_borges)
vocab_unico_borges = {w for o in oraciones_borges for w in o}

print(f"Oraciones: {len(oraciones_borges)}")
print(f"Palabras totales: {n_palabras}")
print(f"Palabras únicas: {len(vocab_unico_borges)}")
print(f"\nEjemplo de oración tokenizada:")
print(oraciones_borges[3])

## Entrenar Word2Vec con gensim (skip-gram)

Parámetros principales:

| Parámetro | Valor | Significado |
|-----------|-------|-------------|
| `vector_size` | 50 | Cuántos números tiene cada palabra (dimensión del vector) |
| `window` | 5 | Cuántas vecinas mira a cada lado (ventana de contexto) |
| `min_count` | 2 | Ignorar palabras que aparecen menos de 2 veces |
| `sg` | 1 | 1 = skip-gram, 0 = CBOW |
| `epochs` | 80 | Cuántas pasadas sobre el corpus (corpus chico → más épocas) |

Con un solo cuento el modelo aprende **el vocabulario de Borges**, no el español entero.


In [ ]:
modelo_borges = Word2Vec(
    sentences=oraciones_borges,  # corpus: lista de oraciones, cada una es lista de tokens
    vector_size=50,              # dimensión del embedding (cada palabra = vector de 50 números)
    window=5,                    # ventana de contexto: hasta 5 palabras a izq/der del centro
    min_count=2,                 # ignora palabras que aparecen menos de 2 veces en todo el corpus
    workers=2,                   # hilos en paralelo para acelerar el entrenamiento
    sg=1,                        # 1 = Skip-gram (centro → contexto); 0 = CBOW (contexto → centro)
    epochs=80,                   # cuántas veces recorre todo el corpus para ajustar pesos
    seed=42,                     # semilla para reproducir el mismo resultado al re-entrenar
)

print(f"Vocabulario entrenado: {len(modelo_borges.wv)} palabras")

## Explorar el vocabulario aprendido

Antes de buscar vecinos o sumar vectores, veamos **qué palabras conoce el modelo**.

Palabras que aparecieron menos de `min_count=2` veces fueron descartadas.


In [ ]:
vocab_borges = list(modelo_borges.wv.index_to_key)
print(f"Total de palabras en el vocabulario: {len(vocab_borges)}")
print(f"\nPrimeras 50 palabras:")
print(vocab_borges[:50])

## Ver las palabras más frecuentes

Además del vocabulario del modelo, veamos cuáles aparecen más veces en el cuento.


In [ ]:
freq_borges = Counter(w for o in oraciones_borges for w in o)
print("30 palabras más frecuentes:")
for palabra, cuenta in freq_borges.most_common(30):
    print(f"  {palabra:15s}  {cuenta:3d}")

## Buscar palabras parecidas

`most_similar` busca palabras cuyo vector está **cerca** (coseno alto) en el espacio aprendido.

Probamos con palabras clave del cuento.


In [ ]:
palabras_prueba = ["funes", "memoria", "recuerdo", "tiempo", "noche"]

for w in palabras_prueba:
    if w in modelo_borges.wv:
        print(f"\nMás parecidas a '{w}':")
        for vecina, score in modelo_borges.wv.most_similar(w, topn=6):
            print(f"  {vecina:15s}  {score:.3f}")
    else:
        print(f"\n'{w}' no está en el vocabulario")

## Sumar vectores de palabras

Cada palabra es un vector de 50 números. Puedes **sumarlos** como vectores normales.

La idea: si sumas `"memoria" + "noche"`, el resultado debería estar cerca de palabras que tienen que ver con ambas.

En un corpus grande esto funciona mejor (analogías tipo `rey - hombre + mujer ≈ reina`). En un cuento corto es más ruidoso, pero se ve la mecánica.


In [ ]:
def similitud_coseno(u, v):
    return float(np.dot(u, v) / (np.linalg.norm(u) * np.linalg.norm(v) + 1e-9))

def vecinos_de_vector(modelo, vec, topn=8, excluir=None):
    excluir = set(excluir or [])
    sims = []
    for w in modelo.wv.index_to_key:
        if w in excluir:
            continue
        sims.append((w, similitud_coseno(modelo.wv[w], vec)))
    sims.sort(key=lambda x: -x[1])
    return sims[:topn]

## La mecánica: suma y resta elemento por elemento

Antes de ver ejemplos con palabras del cuento, veamos **exactamente qué operación** hace la computadora cuando sumas o restas vectores.

Cada palabra es un vector de 50 números en nuestro modelo de Borges. La suma/resta funciona **posición por posición**.

Para verlo claro, usemos vectores de **3 dimensiones** (en vez de 50) con el ejemplo clásico: `rey - hombre + mujer`.


Ejemplo con vectores de 3 dimensiones (simplificado):

```
rey    = [ 0.8  0.5 -0.3]
hombre = [ 0.6  0.2 -0.1]
mujer  = [ 0.5 -0.3  0.4]
```

**Paso 1: rey - hombre**

```
Posición 0:  0.8 - 0.6 = 0.2
Posición 1:  0.5 - 0.2 = 0.3
Posición 2: -0.3 - (-0.1) = -0.2

Resultado: [0.2, 0.3, -0.2]
```

**Paso 2: (rey - hombre) + mujer**

```
Posición 0:  0.2 + 0.5 = 0.7
Posición 1:  0.3 + (-0.3) = 0.0
Posición 2: -0.2 + 0.4 = 0.2

Resultado: [0.7, 0.0, 0.2]
```

---

**En una sola línea:**

```
rey - hombre + mujer = [0.7, 0.0, 0.2]
```

Ese vector `[0.7, 0.0, 0.2]` es el que comparamos con todas las palabras del vocabulario para ver cuál está más cerca. En un corpus grande, `reina` suele ser la más cercana.


**Resumen:** cada número del vector se opera con el número en la **misma posición** del otro vector. No hay multiplicación cruzada ni nada complejo — solo suma o resta elemento a elemento.

En Word2Vec real los vectores tienen 50, 100 o más dimensiones, pero la mecánica es idéntica.


## Suma: memoria + noche

Veamos qué palabras están cerca del vector suma.


In [ ]:
if "memoria" in modelo_borges.wv and "noche" in modelo_borges.wv:
    v_suma = modelo_borges.wv["memoria"] + modelo_borges.wv["noche"]
    print("Suma: memoria + noche")
    print("\nPalabras más cercanas al resultado:")
    for w, s in vecinos_de_vector(modelo_borges, v_suma, topn=8, excluir=["memoria", "noche"]):
        print(f"  {w:15s}  {s:.3f}")
else:
    print("Alguna palabra no está en el vocabulario")

## Suma: funes + recuerdo


In [ ]:
if "funes" in modelo_borges.wv and "recuerdo" in modelo_borges.wv:
    v_suma2 = modelo_borges.wv["funes"] + modelo_borges.wv["recuerdo"]
    print("Suma: funes + recuerdo")
    print("\nPalabras más cercanas:")
    for w, s in vecinos_de_vector(modelo_borges, v_suma2, topn=8, excluir=["funes", "recuerdo"]):
        print(f"  {w:15s}  {s:.3f}")

## Resta: funes - hombre

Así como sumas, puedes restar. La analogía clásica es `rey - hombre + mujer ≈ reina`.

En este cuento no tenemos esas palabras, pero probamos con lo que hay.


In [ ]:
if "funes" in modelo_borges.wv and "hombre" in modelo_borges.wv:
    v_resta = modelo_borges.wv["funes"] - modelo_borges.wv["hombre"]
    print("Resta: funes - hombre")
    print("\nPalabras más cercanas:")
    for w, s in vecinos_de_vector(modelo_borges, v_resta, topn=8, excluir=["funes", "hombre"]):
        print(f"  {w:15s}  {s:.3f}")

## ¿Puedo multiplicar o dividir vectores?

**Sí técnicamente**, pero **no tiene sentido semántico** en Word2Vec.

- **Suma y resta** funcionan porque preservan direcciones: `rey - hombre + mujer` mueve el vector en la dirección de "género femenino".
- **Multiplicar por un escalar** (por ejemplo `2 * rey`) solo **alarga el vector** sin cambiar su dirección. El vecino más cercano sigue siendo el mismo.
- **Multiplicar elemento a elemento** o **dividir** entre vectores no tiene interpretación clara en el espacio semántico.

Lo estándar es **suma y resta**; multiplicar/dividir no se usa.


## Demostración: multiplicar por escalar no cambia vecinos


In [ ]:
if "funes" in modelo_borges.wv:
    v_original = modelo_borges.wv["funes"]
    v_doble = 2 * v_original
    
    print("Vecinos de 'funes' (vector original):")
    for w, s in modelo_borges.wv.most_similar("funes", topn=5):
        print(f"  {w:15s}  {s:.3f}")
    
    print("\nVecinos de 'funes × 2' (vector multiplicado):")
    for w, s in vecinos_de_vector(modelo_borges, v_doble, topn=5, excluir=["funes"]):
        print(f"  {w:15s}  {s:.3f}")
    
    print("\nLos vecinos son los mismos porque la dirección no cambió.")

## Visualizar el cuento en 3D con PCA

Los vectores tienen 50 dimensiones. PCA los comprime a 3 para ver vecindades.

Elegimos palabras frecuentes del cuento (sin stopwords) para que el gráfico sea legible.


In [ ]:
palabras_plot = [w for w, c in freq_borges.most_common(80) if w not in STOP and w in modelo_borges.wv][:40]
vectores = np.array([modelo_borges.wv[w] for w in palabras_plot])
coords = PCA(n_components=3, random_state=42).fit_transform(vectores)

fig = go.Figure()
fig.add_trace(go.Scatter3d(
    x=coords[:, 0], y=coords[:, 1], z=coords[:, 2],
    mode="markers+text",
    text=palabras_plot,
    textposition="top center",
    marker=dict(size=6, color="#2563EB", opacity=0.9),
    name="Funes",
))
fig.update_layout(
    title="Borges — Word2Vec (skip-gram) proyectado a 3D con PCA",
    scene=dict(xaxis_title="PC1", yaxis_title="PC2", zaxis_title="PC3"),
    width=900, height=700,
)
fig.show()

**Qué deberías ver:** palabras que aparecen juntas en el cuento (por ejemplo alrededor de *funes*, *recuerdo*, *memoria*) más cerca que palabras de otros temas.

Con **un solo cuento** el mapa es pequeño y frágil. Sirve para ver **tu texto**, no para representar todo el español.


# Parte 2 — Corpus completo en español (Kaggle)

Aquí entrenamos con **todos los `.zip` que haya en la carpeta de Drive del curso** (fragmentos del [corpus de 120M palabras en español](https://www.kaggle.com/datasets/rtatman/120-million-word-spanish-corpus/data)). No mantenemos una lista fija en código: si suben más archivos al Drive, el notebook los detecta.

Descarga, carga, entrenamiento y exploración viven en este notebook.

**Más texto → mejores embeddings** (The Bitter Lesson).


## The Bitter Lesson

Richard Sutton (investigador de RL) resume una idea que en NLP se ve clarísimo:

> **No ganan los trucos de limpieza más ingeniosos, sino los métodos simples con MUCHOS datos y MUCHO cómputo.**

Aquí: limpiar líneas ayuda, pero el salto real viene al pasar de **2 700 palabras** (Borges) a **cientos de miles de oraciones** del corpus español completo.


## Descargar y localizar los ZIP del corpus

Los nombres de archivo **no están hardcodeados**: se leen de la carpeta pública de [Google Drive](https://drive.google.com/drive/folders/1U9V2Wp-ccTAz7JP__FxU1uAF79by-9uf?usp=sharing) con `gdown` (solo los `.zip` que haya ahí).

Con `DESCARGAR_FALTANTES = True` se bajan los que falten en `corpus_descargado/`. Para demo rápida: `N_ARCHIVOS = 3`. Para todo lo del Drive: `N_ARCHIVOS = None`.


In [ ]:
DRIVE_FOLDER_ID = "1U9V2Wp-ccTAz7JP__FxU1uAF79by-9uf"

N_ARCHIVOS = None          # None = todos los .zip del Drive; ej. 3 para demo rápida
DESCARGAR_FALTANTES = True # False si ya tienes todos los zip en corpus_descargado/

def listar_zips_en_drive(folder_id=DRIVE_FOLDER_ID):
    """Devuelve [(nombre_archivo, file_id), ...] de los .zip en la carpeta de Drive."""
    entradas = gdown.download_folder(id=folder_id, skip_download=True, quiet=True)
    if not entradas:
        return []
    zips = []
    for f in entradas:
        nombre = Path(f.path).name
        if nombre.lower().endswith(".zip"):
            zips.append((nombre, f.id))
    return sorted(zips, key=lambda x: x[0])

def descargar_zips_faltantes(n_archivos=None, carpeta=CORPUS_DIR):
    try:
        catalogo = listar_zips_en_drive()
    except Exception as e:
        print(f"No se pudo listar Drive: {e}")
        catalogo = []

    if catalogo:
        print(f"ZIPs en Drive: {len(catalogo)}")
        for nombre, _ in catalogo[:8]:
            print(f"  - {nombre}")
        if len(catalogo) > 8:
            print(f"  ... y {len(catalogo) - 8} más")
    else:
        print("Sin catálogo de Drive; se usarán solo ZIPs locales.")

    if n_archivos is not None:
        catalogo = catalogo[:n_archivos]

    descargados = []
    for i, (nombre, file_id) in enumerate(catalogo, 1):
        destino = carpeta / nombre
        if destino.exists():
            print(f"[{i}/{len(catalogo)}] Ya existe: {nombre}")
            descargados.append(destino)
            continue
        if not DESCARGAR_FALTANTES:
            print(f"[{i}/{len(catalogo)}] Falta (descarga manual): {nombre}")
            continue
        try:
            print(f"[{i}/{len(catalogo)}] Descargando {nombre}...")
            gdown.download(id=file_id, output=str(destino), quiet=True)
            descargados.append(destino)
            print(f"             OK ({destino.stat().st_size / 1e6:.1f} MB)")
        except Exception as e:
            print(f"             Error: {e}")
    return descargados

descargar_zips_faltantes(N_ARCHIVOS)

# Unir Drive + zips locales (corpus_descargado, corpus_kaggle, raíz del módulo)
archivos_zip = sorted({z.resolve() for z in listar_zips_corpus()})
print(f"\nTotal de ZIPs a procesar: {len(archivos_zip)}")

## Leer y tokenizar **todos** los ZIP

Cada `.zip` contiene un archivo de texto (formato XML por líneas). Saltamos etiquetas y `ENDOFARTICLE`. Misma limpieza que Borges: minúsculas, solo letras españolas, oraciones con al menos `MIN_TOKENS` palabras.

`MAX_LINEAS_POR_ARCHIVO = None` lee **cada archivo completo** (corpus entero). Para una demo corta en laptop, pon por ejemplo `80_000`.


In [ ]:
MIN_TOKENS = 5
MAX_LINEAS_POR_ARCHIVO = None  # None = archivo completo; ej. 80_000 para prueba rápida

def _linea_a_oracion(linea):
    linea = linea.strip().lower()
    if not linea or linea.startswith("<") or "endofarticle" in linea:
        return None
    linea = re.sub(r"[^a-záéíóúüñ ]+", " ", linea)
    linea = re.sub(r"\s+", " ", linea).strip()
    if not linea:
        return None
    tokens = linea.split()
    return tokens if len(tokens) >= MIN_TOKENS else None

def cargar_corpus_desde_zips(archivos_zip, max_lineas_por_archivo=MAX_LINEAS_POR_ARCHIVO):
    oraciones = []
    for i, zip_path in enumerate(archivos_zip, 1):
        print(f"[{i}/{len(archivos_zip)}] {zip_path.name}")
        try:
            with zipfile.ZipFile(zip_path, "r") as zf:
                nombre_interno = zf.namelist()[0]
                with zf.open(nombre_interno) as f:
                    for j, raw in enumerate(f):
                        if max_lineas_por_archivo is not None and j >= max_lineas_por_archivo:
                            break
                        tokens = _linea_a_oracion(raw.decode("utf-8", errors="ignore"))
                        if tokens:
                            oraciones.append(tokens)
            print(f"             {len(oraciones):,} oraciones acumuladas")
        except Exception as e:
            print(f"             Error: {e}")
    return oraciones

In [ ]:
if not archivos_zip:
    raise FileNotFoundError(
        "No hay ZIPs del corpus. Pon DESCARGAR_FALTANTES=True, baja archivos del Drive "
        "a corpus_descargado/, o deja spanishText_*.zip en esta carpeta."
    )

oraciones_es = cargar_corpus_desde_zips(archivos_zip)
print(f"\nOraciones cargadas (total): {len(oraciones_es):,}")
print(f"Tokens aproximados: {sum(len(o) for o in oraciones_es):,}")
print("\nEjemplo:")
print(oraciones_es[100][:20])

## Entrenar CBOW y Skip-gram

Entrenamos dos modelos:

- `sg=0` → **CBOW** (contexto → palabra central)
- `sg=1` → **Skip-gram** (palabra central → contexto)

`min_count=10` descarta palabras muy raras (aparecen menos de 10 veces).


In [ ]:
VECTOR_SIZE = 100
WINDOW = 5
MIN_COUNT = 10
EPOCHS = 5

print("Entrenando CBOW...")
es_cbow = Word2Vec(
    sentences=oraciones_es,
    vector_size=VECTOR_SIZE,
    window=WINDOW,
    min_count=MIN_COUNT,
    workers=6,
    sg=0,
    epochs=EPOCHS,
    seed=42
)

print("\nEntrenando Skip-gram...")
es_sg = Word2Vec(
    sentences=oraciones_es,
    vector_size=VECTOR_SIZE,
    window=WINDOW,
    min_count=MIN_COUNT,
    workers=6,
    sg=1,
    epochs=EPOCHS,
    seed=42
)

print(f"\nVocab CBOW: {len(es_cbow.wv)}")
print(f"Vocab Skip-gram: {len(es_sg.wv)}")

## Guardar modelos en disco

Tras entrenar, los modelos viven en memoria (`es_cbow`, `es_sg`). Para usarlos después (o con `10_word2vec_cli.py`), guárdalos aquí en la **misma carpeta del módulo**:

- `modelo_word2vec_cbow.pkl`
- `modelo_word2vec_sg.pkl`

Pon `GUARDAR_MODELOS = True` y ejecuta la celda de abajo **una vez** al terminar el entrenamiento.

In [ ]:
GUARDAR_MODELOS = True  # False si no quieres escribir los .pkl en disco

if GUARDAR_MODELOS:
    ruta_cbow = BASE_DIR / "modelo_word2vec_cbow.pkl"
    ruta_sg = BASE_DIR / "modelo_word2vec_sg.pkl"
    with open(ruta_cbow, "wb") as f:
        pickle.dump(es_cbow, f)
    with open(ruta_sg, "wb") as f:
        pickle.dump(es_sg, f)
    print(f"Guardado CBOW:     {ruta_cbow.resolve()}")
    print(f"Guardado Skip-gram: {ruta_sg.resolve()}")
else:
    print("GUARDAR_MODELOS=False — solo en memoria (es_cbow, es_sg)")

## Explorar vocabulario del corpus

Veamos qué palabras tiene el modelo entrenado con el corpus grande.


In [ ]:
vocab_corpus = list(es_sg.wv.index_to_key)
print(f"Total de palabras en vocabulario: {len(vocab_corpus)}")
print(f"\nPrimeras 60 palabras:")
print(vocab_corpus[:60])

## Vecinos más cercanos

Probamos con palabras comunes en español.

Si alguna no está en el vocabulario (`min_count` o falta de acento en el corpus), lo verás como OOV (out of vocabulary).


In [ ]:
def mostrar_similares(modelo, palabras, titulo, topn=7):
    print(f"\n===== {titulo} =====\n")
    for w in palabras:
        if w in modelo.wv:
            print(f"Más similares a '{w}':")
            for vecina, score in modelo.wv.most_similar(w, topn=topn):
                print(f"  {vecina:15s}  {score:.4f}")
            print()
        else:
            print(f"'{w}' no está en el vocabulario\n")

In [ ]:
objetivo = ["rey", "reina", "hombre", "mujer", "madrid", "barcelona"]
mostrar_similares(es_cbow, objetivo, "CBOW")

In [ ]:
mostrar_similares(es_sg, objetivo, "Skip-gram")

## Similitudes entre pares de palabras

El coseno mide qué tan parecidos son dos vectores. Valores cerca de 1 = muy parecidos.


In [ ]:
pares = [("rey", "reina"), ("hombre", "mujer"), ("paz", "guerra"), ("madrid", "barcelona")]

print("Similitudes (CBOW):\n")
for a, b in pares:
    if a in es_cbow.wv and b in es_cbow.wv:
        print(f"  sim({a}, {b}) = {es_cbow.wv.similarity(a, b):.4f}")
    else:
        print(f"  Falta {a} o {b}")

## Analogías

La analogía clásica: `rey - hombre + mujer ≈ reina`.


In [ ]:
def mostrar_analogia(modelo, positivos, negativos, titulo, topn=6):
    print(f"\n=== {titulo} ===\n")
    print(f"Positivos: {positivos}")
    print(f"Negativos: {negativos}\n")
    try:
        for w, s in modelo.wv.most_similar(positive=positivos, negative=negativos, topn=topn):
            print(f"  {w:15s}  {s:.4f}")
    except KeyError as e:
        print(f"Palabra fuera del vocabulario: {e}")

In [ ]:
mostrar_analogia(es_cbow, ["rey", "mujer"], ["hombre"], "CBOW: rey - hombre + mujer")

In [ ]:
mostrar_analogia(es_sg, ["rey", "mujer"], ["hombre"], "Skip-gram: rey - hombre + mujer")

In [ ]:
mostrar_analogia(es_cbow, ["padre", "mujer"], ["hombre"], "CBOW: padre - hombre + mujer")

## Ver el embedding crudo de una palabra

Hasta ahora hemos visto vecinos, sumas y restas. Pero **¿cómo se ve el vector en sí?**

Cada palabra en el modelo es un array de números (100 dimensiones en este caso). Veamos el vector de `"rey"` completo.


In [ ]:
if "rey" in es_sg.wv:
    vector_rey = es_sg.wv["rey"]
    print(f"Vector de 'rey' (Skip-gram):")
    print(f"Shape: {vector_rey.shape}")
    print(vector_rey)

Esos 100 números **son la representación** de `"rey"` que el modelo aprendió.

Cuando haces `rey - hombre`, estás restando 100 números menos 100 números. Cuando buscas vecinos, comparas esos 100 números con los 100 números de todas las demás palabras usando coseno.

Los números individuales **no tienen significado directo** (no es que "la posición 5 = género" o algo así). El significado está en la **geometría**: palabras con vectores parecidos (coseno alto) tienen significados parecidos.


## Suma explícita: rey + reina

Sumamos los vectores directamente y buscamos vecinos del resultado.


In [ ]:
if "rey" in es_sg.wv and "reina" in es_sg.wv:
    v = es_sg.wv["rey"] + es_sg.wv["reina"]
    print("Suma: rey + reina")
    print("\nVecinos más cercanos:")
    for w, s in vecinos_de_vector(es_sg, v, topn=8, excluir=["rey", "reina"]):
        print(f"  {w:15s}  {s:.4f}")

## Suma aleatoria del corpus grande

Tomemos dos palabras al azar (sin stopwords) y veamos qué da su suma.


In [ ]:
vocab_corpus_filtrado = [w for w in vocab_corpus if w not in STOP and len(w) > 3]
p1, p2 = random.sample(vocab_corpus_filtrado, 2)
v_random = es_sg.wv[p1] + es_sg.wv[p2]

print(f"Suma aleatoria: {p1} + {p2}")
print("\nPalabras más cercanas:")
for w, s in vecinos_de_vector(es_sg, v_random, topn=8, excluir=[p1, p2]):
    print(f"  {w:15s}  {s:.4f}")

## Otra suma aleatoria


In [ ]:
p1, p2 = random.sample(vocab_corpus_filtrado, 2)
v_random = es_sg.wv[p1] + es_sg.wv[p2]

print(f"Suma aleatoria: {p1} + {p2}")
print("\nPalabras más cercanas:")
for w, s in vecinos_de_vector(es_sg, v_random, topn=8, excluir=[p1, p2]):
    print(f"  {w:15s}  {s:.4f}")

## Resta aleatoria


In [ ]:
p1, p2 = random.sample(vocab_corpus_filtrado, 2)
v_random = es_sg.wv[p1] - es_sg.wv[p2]

print(f"Resta aleatoria: {p1} - {p2}")
print("\nPalabras más cercanas:")
for w, s in vecinos_de_vector(es_sg, v_random, topn=8, excluir=[p1, p2]):
    print(f"  {w:15s}  {s:.4f}")

## PCA 3D del corpus grande


In [ ]:
# No metemos TODO el vocabulario (miles): el gráfico sería ilegible.
# Tomamos las más frecuentes del corpus (sin stopwords), hasta MAX_PALABRAS.
MAX_PALABRAS_PCA = 80
freq_es = Counter(w for o in oraciones_es for w in o)
palabras_plot_es = [
    w for w, _ in freq_es.most_common(500)
    if w not in STOP and w in es_sg.wv and len(w) > 2
][:MAX_PALABRAS_PCA]

coords_es = PCA(n_components=3, random_state=42).fit_transform(
    np.array([es_sg.wv[w] for w in palabras_plot_es])
)
print(f"Palabras en el gráfico: {len(palabras_plot_es)}")

fig2 = go.Figure()
fig2.add_trace(go.Scatter3d(
    x=coords_es[:, 0], y=coords_es[:, 1], z=coords_es[:, 2],
    mode="markers+text",
    text=palabras_plot_es,
    textposition="top center",
    textfont=dict(size=7),
    marker=dict(size=5, color="#16A34A", opacity=0.85),
))
fig2.update_layout(
    title=f"Corpus español (Skip-gram) — PCA 3D — top {len(palabras_plot_es)} palabras frecuentes",
    scene=dict(xaxis_title="PC1", yaxis_title="PC2", zaxis_title="PC3"),
    width=950, height=750,
)
fig2.show()

## Exportar embeddings para [TensorFlow Embedding Projector](https://projector.tensorflow.org/)

El [projector](https://projector.tensorflow.org/) espera dos archivos TSV:

1. **Vectores** — una fila por palabra; valores separados por tabulador (`\t`).
2. **Metadatos** — una etiqueta por fila (la palabra), en el mismo orden.

La celda siguiente exporta el último modelo entrenado (`es_sg`, Skip-gram del corpus) a `embedding_projector/`:

- `corpus_skipgram_vectors.tsv` — top 500 palabras frecuentes (sin stopwords)
- `corpus_skipgram_metadata.tsv` — etiqueta por fila

**Uso:** en [projector.tensorflow.org](https://projector.tensorflow.org/) → *Load data from your computer* → primero `corpus_skipgram_vectors.tsv`, luego `corpus_skipgram_metadata.tsv`. Explora con t-SNE, UMAP o PCA en el panel derecho.

In [ ]:
def exportar_embedding_projector(modelo, palabras, prefijo, carpeta=None):
    """Escribe *_vectors.tsv y *_metadata.tsv para projector.tensorflow.org."""
    carpeta = carpeta or (BASE_DIR / "embedding_projector")
    carpeta.mkdir(parents=True, exist_ok=True)
    ruta_vectors = carpeta / f"{prefijo}_vectors.tsv"
    ruta_metadata = carpeta / f"{prefijo}_metadata.tsv"

    exportadas = []
    with open(ruta_vectors, "w", encoding="utf-8") as out_v, open(
        ruta_metadata, "w", encoding="utf-8"
    ) as out_m:
        for word in palabras:
            if word not in modelo.wv:
                continue
            vec = modelo.wv[word]
            out_m.write(word + "\n")
            out_v.write("\t".join(f"{x:.6f}" for x in vec) + "\n")
            exportadas.append(word)

    return ruta_vectors, ruta_metadata, exportadas


# Último modelo entrenado: Skip-gram del corpus (es_sg)
MAX_EXPORT_CORPUS = 500
freq_es_export = Counter(w for o in oraciones_es for w in o)
palabras_corpus = [
    w
    for w, _ in freq_es_export.most_common(2000)
    if w not in STOP and len(w) > 2 and w in es_sg.wv
][:MAX_EXPORT_CORPUS]

ruta_vectors, ruta_metadata, palabras_exportadas = exportar_embedding_projector(
    es_sg, palabras_corpus, "corpus_skipgram"
)

print("Archivos listos para https://projector.tensorflow.org/\n")
print(f"Skip-gram — {len(palabras_exportadas)} palabras")
print(f"  vectores:   {ruta_vectors.resolve()}")
print(f"  metadatos:  {ruta_metadata.resolve()}")
print("\nPaso 1: corpus_skipgram_vectors.tsv")
print("Paso 2: corpus_skipgram_metadata.tsv")

# Cierre — qué nos falta y por qué vienen los transformers

| Objetivo | Word2Vec | Siguiente paso |
|----------|----------|----------------|
| Mapa de **tu** cuento/PDF | Parte 1 | Más texto propio |
| Sinónimos, ciudades, analogías | Parte 2 + **volumen** | Más datos Kaggle / Hugging Face |
| Entender **tu dominio** a fondo | Parcial | Corpus del dominio o embeddings preentrenados |
| **Generar** texto o comparar PDFs | No | **Transformers** (BERT, GPT) |


## Fuentes de corpus en español

- [Kaggle — 120M Spanish Corpus](https://www.kaggle.com/datasets/rtatman/120-million-word-spanish-corpus/data) (58 archivos)
- [Hugging Face Datasets](https://huggingface.co/datasets) (buscar `spanish`, `es`)
- Modelos preentrenados: fastText `cc.es.300.bin`, Spanish 3B words en Zenodo


## Resumen: The Bitter Lesson

1. **Un cuento** → pipeline claro, mapa personal, poca semántica general.
2. **Muchas oraciones** → relaciones estables (rey↔reina, ciudades, familia).
3. **Limpiar ayuda; el volumen manda.**

Cuando quieras que la IA **lea tu corpus con contexto** o **escriba texto nuevo**, pasamos a transformers. Word2Vec sigue siendo la base para entender qué es un **embedding**.


## Experimentos opcionales

1. Cambia `N_ARCHIVOS` (3 vs `None` = todo el Drive) y compara `most_similar('rey')`.
2. `MAX_LINEAS_POR_ARCHIVO = 80_000` vs `None` — tiempo vs calidad.
3. Cambia `sg` solo en Borges (0 vs 1) y compara el PCA.
4. Sube `EPOCHS` o `VECTOR_SIZE` en la Parte 2 y vuelve a entrenar.
5. `modelo_borges.build_vocab(nuevas, update=True)` con texto propio de alumnos.
